# BirdCLEF+ 2026 - Perch v2 + LightGBM
1. Perch로 train audio 임베딩 추출
2. LightGBM 학습
3. Test soundscape 임베딩 추출 → 예측


In [ ]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import librosa
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
import tensorflow_hub as hub

SR = 32000
SEGMENT = 5

# Competition 데이터 경로
for p in ['/kaggle/input/birdclef-2026', '/kaggle/input/competitions/birdclef-2026']:
    if os.path.exists(p):
        COMP = p
        break
print(f'COMP: {COMP}')

# 사전 추출 임베딩 경로
EMB_PATH = '/kaggle/input/google-perch-embeddings/perch_embeddings'
if not os.path.exists(EMB_PATH):
    # 다른 경로 시도
    import glob
    npy = glob.glob('/kaggle/input/**/audio_embeddings.npy', recursive=True)
    if npy:
        EMB_PATH = os.path.dirname(npy[0])
    else:
        print('=== /kaggle/input ===')
        for item in sorted(os.listdir('/kaggle/input')):
            print(f'  {item}')
        raise RuntimeError('Embeddings not found!')
print(f'EMB: {EMB_PATH}')

# Perch 모델 (test 추론용)
import glob
perch_pb = glob.glob('/kaggle/input/**/saved_model.pb', recursive=True)
model_dir = os.path.dirname(perch_pb[0])
print(f'Perch: {model_dir}')
perch_model = hub.load(model_dir)
print('All loaded.')

In [ ]:
def extract_embedding(audio, sr=32000):
    """5초 오디오 → Perch 임베딩 (1536차원)"""
    if len(audio) < sr * SEGMENT:
        audio = np.pad(audio, (0, sr * SEGMENT - len(audio)))
    audio = audio[:sr * SEGMENT].astype(np.float32)
    logits, embeddings = perch_model.infer_tf(audio[np.newaxis, :])
    return embeddings.numpy().mean(axis=0)

print('Extract function ready.')

In [ ]:
# 사전 추출 임베딩 로드 (변환 과정 스킵!)
taxonomy = pd.read_csv(f'{COMP}/taxonomy.csv')
species_cols = taxonomy['primary_label'].astype(str).tolist()
label_to_idx = {s: i for i, s in enumerate(species_cols)}

X_audio = np.load(f'{EMB_PATH}/audio_embeddings.npy')
meta_audio = pd.read_csv(f'{EMB_PATH}/audio_metadata.csv')
X_ss = np.load(f'{EMB_PATH}/ss_embeddings.npy')
meta_ss = pd.read_csv(f'{EMB_PATH}/ss_metadata.csv')

print(f'Species: {len(species_cols)}')
print(f'Audio: {X_audio.shape}, Soundscape: {X_ss.shape}')

In [ ]:
# 임베딩 이미 로드됨 → 스킵
print('Pre-extracted embeddings loaded. Skip extraction.')

In [ ]:
# 임베딩 이미 로드됨 → 스킵
print('Soundscape embeddings already loaded.')

In [ ]:
# Labels
y_audio = np.zeros((len(X_audio), len(species_cols)), dtype=np.float32)
for i, row in meta_audio.iterrows():
    if str(row['label']) in label_to_idx:
        y_audio[i, label_to_idx[str(row['label'])]] = 1.0

y_ss = np.zeros((len(X_ss), len(species_cols)), dtype=np.float32)
for i, row in meta_ss.iterrows():
    for label in str(row['label']).split(';'):
        if label.strip() in label_to_idx:
            y_ss[i, label_to_idx[label.strip()]] = 1.0

X_all = np.vstack([X_audio, X_ss])
y_all = np.vstack([y_audio, y_ss])
print(f'Combined: {X_all.shape}')

In [ ]:
# Train XGBoost per species (trial_003에서 XGBoost가 LightGBM보다 좋았음)
print('Training XGBoost...')
models = {}
for j, sp in enumerate(species_cols):
    y_j = y_all[:, j]
    if y_j.sum() == 0:
        continue
    spw = max(1, (y_j == 0).sum() / max(1, (y_j == 1).sum()))
    m = xgb.XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4,
                           random_state=42, verbosity=0, scale_pos_weight=spw)
    m.fit(X_all, y_j)
    models[j] = m
    if (j+1) % 50 == 0:
        print(f'  {j+1}/{len(species_cols)} species')
print(f'Trained {len(models)} models.')

In [ ]:
# Predict test soundscapes
print('Predicting test soundscapes...')
test_dir = f'{COMP}/test_soundscapes'
test_files = sorted([f for f in os.listdir(test_dir) if f.endswith('.ogg')])
print(f'Test files: {len(test_files)}')

results = []
for fname in test_files:
    path = os.path.join(test_dir, fname)
    y, _ = librosa.load(path, sr=SR)
    n_segments = len(y) // (SR * SEGMENT)
    
    for seg in range(n_segments):
        start = seg * SR * SEGMENT
        chunk = y[start:start + SR * SEGMENT]
        emb = extract_embedding(chunk, SR)
        
        row = {'row_id': f'{fname.replace(".ogg", "")}_{seg * SEGMENT}'}
        for j, sp in enumerate(species_cols):
            if j in models:
                row[sp] = float(models[j].predict_proba(emb.reshape(1, -1))[:, 1][0])
            else:
                row[sp] = 0.0
        results.append(row)

submission = pd.DataFrame(results)
print(f'Submission shape: {submission.shape}')

In [ ]:
# Save
sample = pd.read_csv(f'{COMP}/sample_submission.csv')
submission = sample[['row_id']].merge(submission, on='row_id', how='left').fillna(0.0)
submission.to_csv('submission.csv', index=False)
print(f'Saved submission.csv: {submission.shape}')
submission.head()